[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ratchanon12/2306565/blob/main/boiler_intro.ipynb)

# What is a database? — hands-on notebook

**One month of data from a boiler house · April 2026**

You do not need any experience with databases to use this notebook. Run the cells from the
top. Where you see **`# YOUR TURN`**, change the query and run the cell again — you cannot
break anything, and if you get an error just fix it and run it again.

> ### Working in Google Colab? Do this first
> **File → Save a copy in Drive.** That gives you your own copy, saved to your Google
> account, so the answers you type today are still there tomorrow. If you skip it, your work
> disappears when you close the tab.
>
> Run cells with the ▶ button on the left of each cell, or **Shift + Enter**.

### The five tables

| Table | One row is | Rows |
|---|---|---|
| `boilers` | one boiler | 3 |
| `operators` | one person | 6 |
| `readings` | one instrument reading, every 4 hours | 540 |
| `daily_logs` | one boiler on one day | 90 |
| `water_tests` | one weekly water sample | 15 |

### The shape of every query you will write today

```
SELECT   which columns you want
FROM     which table
JOIN     another table  ON  how they match      (only when you need it)
WHERE    which rows you want                    (optional)
GROUP BY what to summarise by                   (optional)
ORDER BY how to sort                            (optional)
LIMIT    how many rows to show                  (optional)
```

Only `SELECT` and `FROM` are compulsory.

## Step 1 — set the database up

Run the next two cells. The first one **builds the database for you** — there is nothing to
upload and nothing to install. The second one connects to it and shows you the first table.

> If Colab disconnects or you come back tomorrow, just run these two cells again.

In [ ]:
# ---- RUN ME FIRST -------------------------------------------------------
# This builds boilerhouse.db. You do not need to read or understand this cell.
import base64, gzip, sqlite3, os

_DATA = """
H4sIADmkjWoC/51dW2+dx5F89684CLCQA9DETM89eZIVxtFGkg1J2cRPAmPREjcSKVAUsv73O5fvMl2HySlHwCbKil9zai41
Pd3TNd9efPf0xeH1y8cvXj1+8vrp9y9+/9WTlxePX18cXj/+9tnF4e+31x+u7j4fvv7qUP+M//Xm+m39+9MXry++u3h5+OHl
0+ePX/54+PPFj2f9h366fXt1WP68vvjb68OL7+v//eXZs/HPN5cf/90/313eX9+8e3P/6f3hUBvybP/n8ck33xzeXn2+fndz
+Hx/dfnxcPvl/tOX+7PD/e3NzdXh09Xd4f3tl7tu6ucvVx8e/E1f/fb3Xz198eri5euG4vvDbxaUvzn8z+Nnf7l49bU9e/Tt
N8Y+qv/V/+VQ/+rCuTl79OLy/svd5YfDu8vPj05ZkW5Fdivyn1hx3YrbrdS/OvOAFTVuby+vP/zy5sPtu3Xo6l/HuP27oXto
fNdOq6Pxx4uXFy+eXLxaZ8XX28//dnx/W/v/8v62W/h3368/9/nr6YvFRmvo28v7q+P5sU2ARz/WP988f/7NH/7wqH/Tp8Kb
+/7v/2LSjNny8bLOzfv3l/e1g35ZJs02V968+99jC4fNQu3mw9+/3N3cTwa++++jubT3/DSd7FmdDGIkfmN8n1gxm/NyJibJ
uSMsSP3egQXnzt2ZLamcG8KCq997bSEkV6eRTcGde8KCryjmNtTZnIxrKKwXykKoKLy2EH06r51jxJ0LYSHWFgSw4GxDUeqa
yISFVFHMbairKZZwHisKG88LYSFXFAEs+DhQcG0oFUUECza10cy5nFtmRlXEqhG+wUh9MGpXMjBsm5MRTARzXqeacf48MCba
pLTaRKidGSoQ4aaEbQbmVoS2NlIbD1PkPDImfAViwURM5/X/bWp3UkBCBSJgwkqbV7muUGZMbTyzqhWxLg9b+ogUSy1Rm85E
tSK2meVqd4qxgeIJm8+cWqTNhDG1L2x2iVpitgBbpTYi7jy17jTU1BIDdNVMRNOBxHKeGBMW+KqaCLkzXsqea4UAYWVFWExf
iAPGyg2I9DWSAtcKD5TVTDSmaYs9cSYCcFZpIyJj/8jUMpMIpFUGkEZaKVJrRBKwVmmL3fapZQoHJGvWsqYBCX1e5MiNSNGs
1U24tpf2nZDpC2c0a3UT1vURqd3J9IWzmrWsbYs996mVM8XgTjRrNRPRmzrBxZDd6ZxmrWYilN4X2WSqO53XrGXbhloHtdq1
httHXNCs1U3kvthtpRzKN4matboJ22Znp1+qFUmzlm17ah2KtgkUjjtd1qzVTYivxGdL5DYBVzRrNRNtjbQRsZbzsoxmLevb
1JLenTFTG6K3mrWaiZg6a1nLdacXzVq27+zxPLedPVLd6Z1mLdt39rhuy8y88F6zVjfhbTdhuanlg2atZiKk5hzYFCM1qD4C
a7Wdva6v5jKGSO1mPgFrxeG/N9YqnpsXGVgrjjXSRqRuRQzl+AKslbZ5YTy3mwUDrJWGA153qFJ3M6Y7gwXWaiaM606j57oz
CLBWHvtIn1okEAes1UyEPDZE8jTjgbXy8LUaa7nImQjAWm1brttY834jtxWFCKzVTIRxHCEneEjAWs2EHUAit8xC1qwlbU+t
rl4bEdLvDEWzVjeRbPM72ZNZNJq1ugkTuxsfPMVa0WrWkrazVxN9sRtqRKJo1momoozFHrm+iE6zlljlazGbQPSatWTZ2e3Y
2SkgQbNWNxHbqaguM86/iFGzVjMRctvNqhsvXCuSZi1x6jzCTPCYNWt1E2LXnZ3hzlg0a0nf2VNfI8INajKataTt7JUvcnOU
uG05Wc1azUSUttir98ut1CSataTv7DJcFG4rSk6zloRxsGpTy3Gua/KataSf2XP3cqqjRAVygmatbqJuy22xR26lpgisFcca
aSNiAtcXCVir7+zSp1bhVmrKwFqxrZHcd3ZjqQ0xFWCtNHyt3LxfQ03wbIC1mgnf4xemciczItkCa7Uzeykj+OCpNZIFWCuP
KIqMgxUVnnPAWn1n7/uIiZaaF9kDa+U2In2xp8K5KDkAa5UGJPUJ3siLMRGBtcqIorjhQDNTKydgrWaizilpI1K4Qc2atZwZ
U6tvAokDUjRrdRN1Q8yNtbipVYxmrWaijUhqIzIcJZX+2BIMS/bjocTEUfbj5y8fPrxZ0lMPZKY+v7/++f6hdNKeInh8dvj2
cHt3eHKUFdgaNOeYXt1+/On95fXh1d315y//eHT26PGj099JS/zcXV0d/np78478pvbc45vLm8O3t7c3H6+uWi6J+KoSyg/X
Hw9PahN/ufx0e3dDflfX75+v7++vP9UGHp7e3L+/rD9Qv31CfFuXzF8vb67fXh5+eH93+/Hj5fqdGt67q8u31zdbbmv5nyfy
knNy6z/Kbt1/Pux/HkhN6czU4U9/+t3z5yo/1XKaDyaotvzUzx9u//lgLvPz/eVP/3hzf/Xx05uf/oWFnz98uepJqvZTrV+/
3F2dHd5evXsyloC8+fTTOn8fSHHNFm7/75d3Vzdnh/86msdrz6vc1pRVOhjzO1MXp/T9S0L9z+oYpJNmdOB7N9NDDqn535QZ
HfzezYTGFJK7GRdPmvEIyg8zpW0BEroZcSfNBATlV1At7VYqk1cz+aSZiKD8Csr11tRfc178STMJQeV1pOpeJX60Jpw0kxHU
YiZ1UHkM+EkrBTHlFZNsA3XaSk+DzWasdDM9P77OPne6hy3kZ1c70iZem36lNed031jI0m52fPOvJXc7cnqorENccR2rNo/d
6B45bccjrrgOVuq4MtmegLiinoLdDjFeEXCJWRdWY4t2tOT6OQEu2eii7LhO00VPlz1kJ8zzh+jnonDJxjvd+ZUW9HTn/rSd
njh7yE5qy6IlJ6odc7qfBTL1Ew+29jRPtvbzaQITQVx+nYfS52Hg5qE4xOVXXI3e2yyqFHZ6nfZk2kN2QiNm15LXddzLaTsB
cW0k1tapHXZOz8OeVnvIzsobqc0fwk5CXFnvW8Kt055gm+2s/FP6Ok19HjL9UwDXaqdfUkj9PHZeTu+jPdX2kJ3Q13sZ7Tk9
7s4irjjNn8qujTfc6fnsBHFtPFb6+rLceDmHuOLK89LnIYvLAy7ZvKeybO3unIAVAJZstNHiU2Ys99NmIqBazfhltLrbc9qM
djTc5BKGX0OGTnsabvIJ0+4TEp2sfY3dztiUU7fjT3ePN4hrI8NB8oUjZ28R1+YWhr64upN6evPygrg2MgydNNrkkdP97B3i
ytrbLZzv7T3iyvPmxU4fHxBW1rDannx6ifoIqHYqlE4ZbRba05TqE6DaqdDsa4toTwZYOxUOyqiz/TwT3VMQ1+YaDlesU89p
ag4GccWJM1YqPG3GIqw4O/Kp71yBaI4ArN0zHMPV7NjTO1dwAEt7hj1re54IOx5w7VyYOhd2T+z0cAXtafjJMzQbGRI7V9Ce
hp88w+HxBm4aBu1p+IkM83ZCIU7IISOuzaMbZBg5jy4UxDXbWT350/0TDeLysweVaDsWcW1kGPryIudhFMQ1e5iVNvrmdbqf
o0Ncs4fZrmK08TpNY9EDrpV+1jhNw+VOb14xAK6dDoerYdq4n94sYgRc+0nZbvPQEP2TEFecN2UZJzhivDLiitM8zLb3M9Oe
grjifKJcxut0PycDuBb+WSIbrp8s/On5nCzg2l3DuIRrHkg9HpsRgLUflHNfXt1HIGBpXyNMdChbZI3wDZP2NcLkG47hsqQd
7WyE6aA8fMPWzYbon4i4tohh3JYX4YulhLj8TPOlL698ehqmjLj8TPPLcifaUxBX1tuypXy6bBDWxobD5W2712kSyxZR5Xlx
FXb2ZLgvDmFD+jiZ4dL45NO10UpjcZ1eFdkDrt3OIHnhejkgrKhPk4ZbpDkirNk1zDIGnbCTEJaKGo5RJw4WOQOu/ZS8+4aE
z5IL4Np9Q7NF14jTfzGAayfDcUIx3OIqUDAwkaHtHC/cXlqgamAiw7iRIRGlK1A6MPmG4+CeyfZ4xOXneejY6FoJiMtPyyKP
aB+RaogIazHjlrCG5VyxkhBWnoLpy5ZMBOlKRlh5imv0ujDKRSgFceU5yJtJUrXGAK79pOw7baS25xTCkAVgu28Y+jxMXEdb
I4Bsdw5TPysXLuNljUNocY6HjjSTYVrkEZpyD4WNRVkTEBpwomdbFAEahg49mbowCaDtHqLZ9mZD5GRMBmgYPnScD2SNdjoS
BBDHQiO6yGqvI01n5rTwYp3YRNbKar8jQTolD2ec6CIriMzP89GP/CCTr3QIzU/+eIUWOYq11iO0zVGMfam1vcwTmTQbEFrW
m7QhU402IrQ8B6b4+WgTQtt41m3eIhHCsTYDtP3sLFsA2TEtKgBtN2T3QybR2WIAmgom9gVHHVusWIQW5/BU4A0JQotT3K3d
0+XCknV5I7So15pjU/EeoO0RxRHJadu+IdaaBIAmyr1aIsCMoQjQdr/Rbn4jEaOqU05ByxNBDgfLsp2tXZE8MeTkOTITUvsi
eWJIt4cDibXmDELz84QcNx8I58g6i9D8tPqXgKlhrmIIQvM6ROAbNGKJOIfQ1AWcQF9acB6hZX0+S2wfBYSmztMjCMfsay4C
tN2FHAdYurMTQNsv4vQL6uPISMxslwGaPlIvcW6mRQWhqUN1ZgOM1huEBgzZZnZgDFmEFo/9LGY78gLQxOi1JiQfeQfQZL62
sgQMGF/Ue4Cm7+PQN+as195IgaN1YM981mtvpEwM6ZebEFRMxXrtjZSJIfed1jKdnRGaP74LIcw8KgjNT5c86oRMJPkHg9D8
FMt3wxAzasEitC3auJ+wqRYJQssTZ+dxz4xZ/cEhtLzGDsoW92Y6O3iAtp+NS89TOJIhQwBo+qriOIsQSy1EQLYTpO1LzZCb
SEiIbHMhS+f+HkIn7GQEFufrHmOlUT1UEFnUu5oh99loANl+xDa7m0XYsYBM5aSzGS4tMYmiADJRAboRyWJOfREEEMCBHBzC
LLSoXJHJUFjoMbEXQ5UrMhla84Fs8CBGhOb1OTSS54eYEJqix7GpUaOWEZqfR22kpilDBaFlHajz5GEtGYS2nYxlc0UoQxah
5RWa26ERm1oSgLY7kHbfHYnFnxxA24/YftvUiMtVNnmAhvzYo6JMiwJCi9rtJ+9k1jMLQptT3UvMh/FEU0Jocc5f0Lcy624D
0DAGmUnKTgWgyRypW6+OEYayAWg71botn0uwSNb6AnbyH9OWSmOCB1mrDFh9o7uyiCe32ay1BqxOzrixZpmzWvYIzeuQD3kx
zuaA0GZHNI9LsMTazxGR+fmqVaI9kZwQWZ6rC5b8J2EnI7Cs8xie3GZzQWSzGypjd2TKOIoBZHvC2m/IqDoFC9CU99ihcZmV
IgBtN2S3uDETXC0OocUpJiojs8KU8BSP0Gb/cRk1ZuMvAaHF+b7wuLLHIIuAbGdHuwf7mQYlQLazo+vIhMwZlQzIdAAydSeL
uFRki/ZE5nqXsMWfmfoSoz0RgdS1YQdNjPZE5ooX2cI0TOWDEYTm16XmutNfqNEX4xCZn0q40jhfMaUzxiMyr6PGwgV7xQRE
lo+LDQhfXUxEaHCbx3BpDDEJoWV9WY4MG4rJAE2nsJcLPYSdAsj2U/pI85LxJ7EGkOlCwMDOImsRWNQJQ/I6qlhBZCqBbchK
E7EOgcWJ95f73sIg84BM+46JLUcVGwDZzo5hzzwRdiIgw1pAR9bwWe2GOCBH2uUTq/2QuQBmRLEK20XaD3FwuB5eMQNNDELz
ui41ULEeEYvI5uTMksEgnFkRQWSqNjr3vg4Eg4hDZHlGtkRViZUvHqHl46iqYfo6IDQokfZkX0dAtp+Iy0YhRDhMJAEynZsR
thJGJAMylZtxS0KVGbSC0OK80pa9mlhpziA0xY5+pJ2IFjmL0CD2yBa6OgFoOjezzEdi13cOoKkT8RJWpfrIAzQxx5U+xKVZ
cdoP8SAZsdwpJiak036In/jRbslCpjhZuyF+8h33AB3Dsy4jso0e4+ZhMbTmCiLzs1scyYJO8QaReZyOXH2yt4gsz1Vnlnav
vSAydbYeO0hgDDmElrVb7LkUj3gP0HShYKJr7n0AaDs/5n4bnLyRJz4CtN153KumA2MoIbSoSSSSxzSfEVqcSlbX0CNhpyAy
oEfyvokEA8jwck/i7uRIsIBMzLHHbwhoQQCaqEqSkZcjDukStCcyl8jEXe6F6SPtiYSJZ01famTpoQTticxFMnbhR7JFEaFB
lYzhsvISEkLzc1TVsee9kBGZKpPp7ymcZ2YaFUSm3Mc4rlESehIGgeWJQ1bHmDFkEVnWHGLJrSgKINvp0SxFAVSuSKIDaLpy
cFTqMSfH6AHa7j622+GW1A6SGBCZEtkZHMJEH2JEZHF2+d2IhhHTOiZEBjI75MUuiRmgYeiRDT7FAtBk1uXKnvYekwFoAnXH
mZzYSXsiEQqqF8kMYj4m7YlEyMyMy7iBYLWkPZG5/MbtrMa0yCM0dTs8kgkVSQGR+XnQRh0Gs9JSRGRe16V57iq2pITI8jxo
wl7HkpQRmqqdGSc+ho1SQWhZJ+U9OWjZALT9eL3vacyazRag7fzYoC3JVGIvygLQVPXMIvTmGUMOocUZGl2tINkjtDjfN6Bj
6jkgsjjfxlp8fqZBEZDJrKawFlwSSy0nQKZTMwMZo9uUAZkojZpx8ZEJQGTtiCRQIlv2IqKLivZEEniPYxZRhrQnkqbgo9sc
CKaviyA08B4jOR2LQ2h+lnQZhpjOLh6heS0S4sjdsQSElo9DIgw/lojQsg5kWTL6WBJCyzqw6kjqLxmg7f5j+DX3saQUgKar
CwPp9jljANnuPrrtvmIgNMCMRWRR+/yZbJAgsDgDi+MQQuhlGYfI4jwdM3vRzBkPyHb3Me4pRwZaAGiq4rq948VdN3AmAjQx
Wt7Fci6NM9oTyRM/rsIIVDWHM9oTyZPXN0iE9NWd0Z5IhqrrhUQI6Vsthmrn0hnfY3TkSnNaDXUytK40S6rtaTnUyZBbCuVp
aA6h5TUpH3Y9VAaaR2hZU7/jImJOK6LaDIqNtLKq05KoNk/+Y9oFhwg7CZDt0cd9UxOmizIg0+4jHTN2WhTVZpDmGZdEGJFE
rYo6GVqdLM9FaZyWRZ0MrTKbXMLAaVlUmyd+zL/mCp3TuqiTobhEHzNXXeK0MOpkyM8yUYQd7YiUyeuTLY5FbPtOK6PaAqfr
5RIV0yLtiJSJHuMWMmaYX2uj2jIlZ8yedmIMFYTm5zOoZ/P7TqujTobUPVxiL9LyqLaA+0jHVZ3WR50MreEeQ+7XWiB1MrSG
e9h5pBVSbYHwY2DPM05rpE6G8nLn3ZMK81oldTLklqNaoNQJnJZJtUW7fatkFIMsI7I4D1oYMgdMiwoii1rXhtR/1UKptoD7
OEiE2WQ9PI4Gurb9ZVwKmYcn0uBuT2ZV2ZzWShUD0hSBLb5zWix1MjSgjatvxPrQaqliwHvkfTWtlypm8h73W6+OGf2EyFTu
eqngZlqUEZrX0MgjqNOSqWLAe1zE6wiVZYPIZi3GvBQFEIMfLCLLx2p6RObBadVUmctmpqsUjIC0A2ha5dazkk1O66ZOhvwm
hFeXLEGzWjhV5moX+RX3cJ0WTp3srB4/qZLstHLqZGhVxyJzhU5Lp8px1QxZLuu0dqoYqCv07BUIp8VTJ0Nr2bUlWU2rp4qF
suvEKtA5LZ8q8B7MklIjguFO66eK1QKPS9kMMfpaP1XsUW6GfCnCaQFVmatmRgaD1WqPCGy++bg+hcA0KCEyeBcmkA3KCCzP
d0TGhV5q7Asim2+Fu1HuQET5ndZQFXskAU7KpDgtojoZyguHFC7n6LSM6mRovbREvpbktI6qWLgXnoajzrTIIzSgR0+OmlZS
FXukSiGcQ6OVVGV+Isb/musvTkupitWqkUsUi4mGai3VydBa5Mru1lpMVeQodc0VJzqtpirwSswS5CceNHBaT3UytG7WjpyO
WlBV5mKXsL8TQ4y+VlQVObrYw775oCVVRUC1Zym/YQwFhKZeR/B0UF2rqopAambxjIhppGVV5bhqJnGxHi2rKnKUue4rljFU
ANkeeuxPf/ZSaYaMtLDqZMjNKsrENNLKqiKQuZYR5WeeDhGEBhfDhYz0aW1VEf1Owqqzx/SRB2hiHlCaJkZNq6uK6DtCOdFr
Vuurylw4s9dgMNkiLbAq88Mxo7qEvLXktMKqOLgYPqAxHraWWBWnc9dLpQJxlvVaY1Wc1vVerpgT7qPXGqvitLzFepUiE4YE
ofnjBK8w0BxCy/OEHDFs5lkcrbE6GYrLtkZqzHutsSoPV84wfR0BmSZIz7694LXEqrij1DX5vKDXEqvitOD4kgZlxqwgsjjL
G4ykPDMdtcTqZGgUciWWjbyWWJ0M+eUIStaUegsvvcMTg57d+L2F995B1Cywb7J5LbEqx0/JGG46aoVV8XC4XnSjmS7Snsj8
lsx+MVyYh6y0J+JB9XFRWGW6KCM0FXwcThbxipDXCqsyF86UPh/Z15q0wupkyE/QmLWvFVZlflCmbJk5htW0wqrML8GY/TxD
2HGITN18zGxVqdcCq3L8pAx5Q8xrgVXBN2WWyjKmryNAw+BjIl960wKrMhfO2MXFrv4j00cZoancTOy5mcL0UUFoUZOIJV82
1gKr8tDDMpSz7rXAqni4+rg4WQT1a4HVyVB/THGJHDCbkRZYlblyxv2aF469FlidDO1XzSg1W68FViUcqVJwJ36v9VVlLpwJ
+/1AZtASIlNXwxcFEGKpaX1VgcoZtzzAw4x+QWjw4FbhBMi91leVuXSm9KVG6gd6ra8qQV99XI5qjIut9VUlwGusfow+0Uda
X3UyNM4zI5vOvGWo9VUnQ+td3MKF1b3WV5UAldfjbU5D2ImILM4C1EOZgNkdtbzqZMgvjM3l+LxWV5Vw9A5hoN7L8lpcdbIz
9rSxYpk9TYurSjiKPrKOsRZXlQin69FDzFlWi6tK1JK4fbumIrRei6vK/N5M2irKiQSv1+KqgoUzdFWI1+KqMlfOrJEsKhHm
tbqqRP0+A3992mt1VYk6arg4WUQyxGt5VWVIOoeQL6Z5La8qEW4+ji2E8US0vKpEuBk+ah0ZVtP6qpOhuNyliOSoaX1VmStn
/Fbw4hhDDqFFXVMuHD1qeVWJQI+lN4h5CzcgMCVpNiIijKuu1VUlQvRx6WqCZ7W66mRoBHuWOxkEG2l11cnQ+l6EJz0ara4q
CSQfee9Rq6vKXDkzpiN569lrdVWZn50ZPj8b69PqqpJAE3cRMmbeQnYIbU7PlBGCYDYjra4qCZ5rXR6YZzo7IDR4VEHIeaTV
VSWB+5jYGwdeq6sKVs4kNorttbqqpCP3kR61AtB0+jqxD5h4ra4qc+mM36QyGOrX8qqS4HLPEhAl+kjLq0oCYYoyRo2YkFpe
VRna5dCZ7VHLq8pcO+O3iyKUoQDQ9O2e4a8FBloEaKj6SKb4vdZXlQzn60Cf+LXAqmSoLUysQrvXAquS4fnWSE9ILbAqeVJ9
tFvqiTk8aoFVyeBBLmc+ApoWWJW5dmbXE2HmkRZYlfnZGbOfaIjh1wKrkkG6Z4lkMp0dEBo8O5PJIK1WWJUMl8M9ewfKa4VV
mYtn1lgW2UcZoB2/XMj5R1phVfIRQ5Kl90ErrMr87IzZRo24+hy0wqpkncFer8EQL8drhVXJwJCJfSkuaIlVweoZP8TWCmHI
A7TdUNhj9Iwh7Y3gszOeVQIJWmJ1MhSX8hnh5FuClliV8kB1IeVoBS2xKnP5zF7z5BhoBaEp7cclJkoY0hqrUvTThaviUiIM
WYSW8VJ/PYwSw69FVqUchSBJ9b+gVVZlLp/Zo/3E0TholVUpcEHcjTgEYScAMvV8TabV1oJWWZ0MrfORzKoGrbIq87MzkyQ6
09cZoUWttefZQSsILeqSUPKeYNAqq1KAIMdJlLhREbTMqhRQ74msmx20zKrMD8/4XbfNEYaUM+LM0btcpLR+0DKrzuiz8ZLI
II5rQcusKkNhU5UgYodB66w6AwTp2dfFg9ZZdQbqC8vIPjHQMkLzulKNfHAwaJ1VNz884/Y7Z4QhrbPqjH6SNY+sOjMhtc6q
myto1qpQJkwftMyqM/C0a2R1BIOWWZ0MjaWWWQXZoGVWnTkqvyZ92qBlVp2Bd7nGfQHPQIsIbX6hoSxP6hAbttZZdfO7M7uS
PdXZGaDpAkNPJnuClllVdkyfjmShYtA6q5Oh+Zm4PvpPXl48fn1xeP3422cXh39e3l/dvbm/+nz/+fD1V4f6p/39zfXb9tf2
yy6+u3h5+OHl0+ePX/54+PPFj2f9h/5+e/2hftd/bP2hF9+/Prz4y7Nnh5cXf7x4efHiycWr5ec+f739/G/P9l/ytv7u+vfX
F397vX08/vnz9Yfrny7ffPr08VCtPX4G//zp/WH7o/75K+ynCd7+dIR6Us0+qoeiNvjmgYTRQ5/rd/3q592PsQ/shg99rR/z
q1/3k2udwI75Wr/gl9vXvv/uwnytn+2rX/dqL3vsfD30sX6rLzfY0jstMl/rQuPQGt6/jhRsXV1cv7b9Sp55oPD2oc91TXH9
PC4j5qnpol0hqb+9SbK2MaNGXEuZ9O/bgwDtP7jfrx2f8f3oPO736+BLafBdgy/UpNFyJe1z21T/28hzzdeRltKbP6Z8o9kn
3z9//vT17/8freB/+gS8AAA=
"""

if os.path.exists("boilerhouse.db"):
    os.remove("boilerhouse.db")

_sql = gzip.decompress(base64.b64decode("".join(_DATA.split()))).decode()
_con = sqlite3.connect("boilerhouse.db")
_con.executescript(_sql)
_con.commit()
_con.close()

print("Database ready:", round(os.path.getsize("boilerhouse.db") / 1024), "KB")
print("Now run the next cell.")

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

con = sqlite3.connect("boilerhouse.db")


def q(sql):
    """Run a SQL query and show the answer as a table."""
    return pd.read_sql_query(sql, con)


# A first look. Change 'boilers' to any other table name and run it again.
q("SELECT * FROM boilers")

That is the whole mechanism: you write SQL between the quotes, and `q(...)` hands you back a
table.

**Two rules that catch everybody:**

* Text goes in single quotes — `code = 'B-02'` is right, `code = "B-02"` is not.
* Numbers do not — `boiler_id = 2`, `rating_tph > 30`.

In [ ]:
# The other small table. Six people, three shifts.
q("SELECT * FROM operators")

In [ ]:
# And the first few rows of the three bigger tables.
display(q("SELECT * FROM readings LIMIT 5"))
display(q("SELECT * FROM daily_logs LIMIT 5"))
display(q("SELECT * FROM water_tests LIMIT 5"))

> **Look carefully at `readings`.** It does not say *Boiler 2* anywhere — it says
> `boiler_id = 2`. To find out what boiler 2 actually is, you look `2` up in the `boilers`
> table. Storing the boiler's details once, in one place, and pointing at them from everywhere
> else, is the whole idea of a database. We will put the two back together in Exercise 3.

---
# Exercise 1 · Looking at the data

`SELECT` · `WHERE` · `ORDER BY` · `LIMIT` · `COUNT`

**Worked example first.** Pick some columns, keep some rows, sort the result:

In [ ]:
q("""
SELECT   code, name, rating_tph
FROM     boilers
WHERE    rating_tph > 30
ORDER BY code
""")

### 1.1 Show the whole `boilers` table.
Then answer out loud: what does one row represent?

In [ ]:
# YOUR TURN 1.1
q("""
SELECT code
FROM   boilers
""")

### 1.2 Show the `name` and `rating_tph` of the boilers rated above 30 t/h.

In [ ]:
# YOUR TURN 1.2
q("""
SELECT *
FROM   boilers
""")

### 1.3 How many rows are in the `readings` table?

*Predict the answer before you run it:* 3 boilers × 6 readings a day × 30 days.

In [ ]:
# YOUR TURN 1.3
q("""
SELECT COUNT(*) AS n_readings
FROM   boilers
""")

### 1.4 Show the five hottest stack temperature readings — all columns, hottest first.

Use `ORDER BY stack_temp_c DESC` and `LIMIT 5`. **Look at which boiler they belong to.**

In [ ]:
# YOUR TURN 1.4
q("""
SELECT *
FROM   readings
LIMIT  5
""")

### 1.5 Show every reading for boiler 2 taken on 1 April 2026.

The `ts` column holds a date *and* a time, so match the start of the text:
`ts LIKE '2026-04-01%'`. Join the two conditions with `AND`.

In [ ]:
# YOUR TURN 1.5
q("""
SELECT *
FROM   readings
WHERE  boiler_id = 2
LIMIT  10
""")

---
# Exercise 2 · Summarising

`AVG` · `MIN` · `MAX` · `SUM` · `GROUP BY`

**Worked example.** One number out of 540 rows — and then one number *per boiler*:

In [ ]:
display(q("""
SELECT ROUND(AVG(steam_tph), 1) AS mean_steam_tph
FROM   readings
"""))

display(q("""
SELECT   boiler_id,
         ROUND(AVG(steam_tph), 1) AS mean_steam_tph,
         COUNT(*)                 AS n_readings
FROM     readings
GROUP BY boiler_id
"""))

`GROUP BY boiler_id` tells the database to deal the rows into piles — one pile per boiler —
and answer your question separately for each pile.

`COUNT(*)` is there as a check: 180 readings went into each average. If one boiler had 40,
you would want to know why *before* trusting the number next to it.

### 2.1 What is the average steam flow across every reading?

In [ ]:
# YOUR TURN 2.1
q("""
SELECT steam_tph
FROM   readings
LIMIT  5
""")

### 2.2 What is the average **stack temperature** for each boiler?

**Look hard at the answer. Is anything odd?**

In [ ]:
# YOUR TURN 2.2
q("""
SELECT   boiler_id
FROM     readings
GROUP BY boiler_id
""")

### 2.3 For each boiler, show the lowest and highest stack temperature.

In [ ]:
# YOUR TURN 2.3
q("""
SELECT   boiler_id,
         ROUND(MIN(stack_temp_c), 1) AS coldest
FROM     readings
GROUP BY boiler_id
""")

### 2.4 How many readings does each boiler have?

In [ ]:
# YOUR TURN 2.4
q("""
SELECT   boiler_id
FROM     readings
GROUP BY boiler_id
""")

### 2.5 From `daily_logs`, the total steam and total gas for each boiler.

Use `SUM(steam_t)` and `SUM(fuel_gj)`.

In [ ]:
# YOUR TURN 2.5
q("""
SELECT   boiler_id
FROM     daily_logs
GROUP BY boiler_id
""")

**A picture of what you just found.** Run this cell once you have done 2.2:

In [ ]:
daily = q("""
SELECT   substr(ts, 1, 10) AS day,
         boiler_id,
         AVG(stack_temp_c) AS stack_c
FROM     readings
GROUP BY day, boiler_id
ORDER BY day
""")

fig, ax = plt.subplots(figsize=(10, 3.6))
for bid, colour in [(1, "#1C7293"), (2, "#E07A3F"), (3, "#0B3C49")]:
    part = daily[daily["boiler_id"] == bid]
    ax.plot(range(len(part)), part["stack_c"], lw=2, color=colour, label=f"boiler {bid}")
ax.set_ylabel("stack temperature, °C")
ax.set_xlabel("day of April")
ax.set_title("Daily average stack temperature")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

---
# Exercise 3 · Joining two tables

`JOIN ... ON`

`readings` only knows `boiler_id = 2`. A **JOIN** is how you look that `2` up in the
`boilers` table and glue the two rows together.

**Worked example.** Read the `ON` line as a sentence: *where these two numbers match, that is
the same boiler.*

In [ ]:
q("""
SELECT b.name, r.ts, r.stack_temp_c
FROM   readings r
JOIN   boilers  b  ON  b.boiler_id = r.boiler_id
LIMIT  10
""")

The letters `r` and `b` are just nicknames, so you can write `r.ts` instead of
`readings.ts`. You choose them yourself.

### 3.1 Show the first 10 readings with the boiler's **name** instead of its number.

In [ ]:
# YOUR TURN 3.1
q("""
SELECT r.ts, r.stack_temp_c
FROM   readings r
LIMIT  10
""")

### 3.2 Show the first 10 daily logs with the boiler **code** and the **operator's name**.

You need two JOINs — one to `boilers`, one to `operators`. Same line, written twice.

In [ ]:
# YOUR TURN 3.2
q("""
SELECT d.log_date, d.steam_t
FROM   daily_logs d
LIMIT  10
""")

### 3.3 Average stack temperature per boiler again — but show the boiler **code** this time.

In [ ]:
# YOUR TURN 3.3
q("""
SELECT   r.boiler_id,
         ROUND(AVG(r.stack_temp_c), 1) AS mean_stack_c
FROM     readings r
GROUP BY r.boiler_id
""")

### 3.4 Gas used per tonne of steam, for each boiler.

Total gas divided by total steam: `SUM(d.fuel_gj) / SUM(d.steam_t)`. Show the boiler code
next to it, and sort so the worst boiler is at the top.

**Which boiler is worst, and by how much?**

In [ ]:
# YOUR TURN 3.4
q("""
SELECT   d.boiler_id,
         ROUND(SUM(d.fuel_gj) / SUM(d.steam_t), 2) AS gj_per_tonne
FROM     daily_logs d
GROUP BY d.boiler_id
""")

### 3.5 No query for this one — just think

Put your answers to **2.2** and **3.4** side by side.

**Why do you think that boiler is using more gas?** Write one sentence in the cell below.

*Hint: where does the heat go if it does not go into the steam?*

In [ ]:
answer = """

  ...your sentence here...

"""
print(answer)

---
## Check yourself

Run this once you have finished. It works out the two numbers and compares them with the
rule of thumb engineers use: **a boiler loses roughly 1 % of its efficiency for every 20 °C
of extra flue gas temperature.**

In [ ]:
stack = q("""
SELECT   b.code, ROUND(AVG(r.stack_temp_c), 1) AS mean_stack_c
FROM     readings r
JOIN     boilers  b ON b.boiler_id = r.boiler_id
GROUP BY b.code
""")

fuel = q("""
SELECT   b.code, ROUND(SUM(d.fuel_gj) / SUM(d.steam_t), 3) AS gj_per_tonne
FROM     daily_logs d
JOIN     boilers    b ON b.boiler_id = d.boiler_id
GROUP BY b.code
""")

both = stack.merge(fuel, on="code")
display(both)

worst = both.loc[both["gj_per_tonne"].idxmax(), "code"]
rest = both[both["code"] != worst]
d_temp = both.loc[both["code"] == worst, "mean_stack_c"].iloc[0] - rest["mean_stack_c"].mean()
d_fuel = both.loc[both["code"] == worst, "gj_per_tonne"].iloc[0] / rest["gj_per_tonne"].mean() - 1

print(f"\n{worst} runs {d_temp:.0f} °C hotter at the stack than the other two.")
print(f"{worst} burns {d_fuel*100:.1f} % more gas per tonne of steam.")
print(f"\nRule of thumb: {d_temp:.0f} °C / 20 = {d_temp/20:.1f} % efficiency lost.")
print(f"Measured from the fuel figures:      {d_fuel*100:.1f} %")
print("\nTwo numbers, from two different tables, that agree.")
print("That is what real evidence looks like — and it means the tubes need cleaning.")

---
## What you learned today

1. **A table is a list of one kind of thing.** If you cannot say what one row is in four
   words, it should probably be two tables.
2. **Store every fact once** and point at it from everywhere else. That is why a database
   stays correct while a spreadsheet slowly drifts.
3. **`GROUP BY` is the one to remember.** Turning 540 rows into one number per boiler is
   what found the problem today.
4. **The computer counts, you interpret.** SQL gave you 183 °C and 3.15 GJ per tonne. Only
   an engineer can say that means the tubes are dirty.

### If you want to keep going

* `NULL` — what a database does about missing readings
* Joining three or more tables at once
* Loading your own CSV file into a table and querying it

### Keeping your work

* **Your answers** are already saved if you did *File → Save a copy in Drive* at the start.
  If you did not, do it now — **File → Save a copy in Drive**.
* **The database file** is rebuilt by the first cell every time, so you never need to keep it.
  But if you would like a copy to open in DB Browser for SQLite at home, run the cell below.

In [ ]:
# Optional: download boilerhouse.db to your own computer (Colab only).
try:
    from google.colab import files
    files.download("boilerhouse.db")
except ImportError:
    print("Not running in Colab — the file is already in this folder:",
          os.path.abspath("boilerhouse.db"))

In [ ]:
con.close()
print("Done. Try asking the database a question nobody set you.")